# MC-dropout — bất định epistemic từ 5 checkpoint đã có

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

**Không train gì cả.** Notebook này chỉ chạy inference: nạp `best.pt` của từng fold,
bật lại dropout, forward `K` lượt trên tập val của chính fold đó, lưu `(K, N, 7)`.

**Vì sao không dùng thẳng 5 checkpoint làm deep ensemble.** Mỗi ca ở val của fold `f`
nằm trong tập train của **cả 4 model kia** (kiểm trên `splits/`, WORKLOG S-080). Gộp
5 model rồi chấm trên 394 ca là để 4/5 thành viên chấm bài họ đã học thuộc. MC-dropout
né đúng chỗ đó: mọi thành viên đều là cùng một model của fold đó, nên đều mù với val.

**Đổi lại:** MC-dropout là xấp xỉ nghèo hơn deep ensemble thật — các thành viên chung
một cực tiểu nên đa dạng ít. Đây là phép đo rẻ để quyết có đáng đốt 4 session Kaggle
cho ensemble nhiều seed hay không, chứ không phải bản thay thế.

**Ngân sách:** ~8 phút GPU cho cả 5 fold ở `K=20`. So với 37.5h của ensemble 3 seed.

**Cần mount hai dataset:** cache E4, và checkpoint (`best-weights`, 5 file `best_fold_N.pt`).

## 0. Bootstrap

Giống notebook 07. Dòng `repo commit` là bằng chứng đang chạy đúng bản code nào.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

# ---- THAM SỐ ---------------------------------------------------------------
FOLDS = [1, 2, 3, 4, 5]
N_PASSES = 20          # số lượt forward mỗi ca
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

os.environ["LLDMMRI_OUTPUT_DIR"] = "/kaggle/working/runs/mc_dropout"
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

CFG_PATH = REPO / "configs" / "baseline_3dpatch.yaml"
CFG = load_yaml(CFG_PATH)
print("dropout_prob trong config:", CFG["model"].get("dropout_prob"))

## 1. Cache E4 và checkpoint

Cần **hai** thứ mount vào: cache E4 (`lesion_tight · 112×112×32 · per_phase`) và 5 file
`best.pt`. Đổi đường dẫn bên dưới cho khớp tên dataset bạn đã upload.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")

# Cache E4 nhận diện bằng NỘI DUNG `cache_meta.json`, không bằng tên dataset. Tên do
# người upload đặt và đã lệch một lần rồi (`lld-mmri-lesion-tight/cache_lesion_tight`
# chứ không phải `lld-mmri-e4-per-phase` như đoán ở S-080). Ba khoá này là thứ phân
# biệt E4 với mọi cache trước đó.
E4_KEYS = {
    "align_phases": "per_phase",          # <- phân biệt E4 với E3
    "target_size": [112, 112, 32],        # <- phân biệt E3/E4 với E0/E1
    "crop_mode": "lesion_tight",          # <- phân biệt E1+ với E0
}

# Hai layout đều gặp trong thực tế, nên dò cả hai thay vì ép một kiểu:
#   A) file phẳng   best_fold_1.pt ... best_fold_5.pt   <- dataset "best weights"
#   B) theo thư mục fold_1/best.pt ...                   <- gói thẳng từ output run
CKPT_PATTERNS = ["best_fold_{f}.pt", "fold_{f}/best.pt", "fold{f}*/best.pt"]

# Quét mọi dataset đang mount thay vì đoán tên — cùng lý do như cache.
CKPT_CANDIDATES = sorted(d for d in INPUT_ROOT.iterdir() if d.is_dir())


def scan_caches(root, max_depth=3):
    """Mọi thư mục dưới /kaggle/input có `cache_meta.json`, kèm nội dung meta."""
    import json as _json

    found = []
    for depth in range(1, max_depth + 1):
        for meta_path in root.glob("/".join(["*"] * depth) + "/cache_meta.json"):
            try:
                meta = _json.loads(meta_path.read_text("utf-8"))
            except Exception as exc:  # noqa: BLE001 - chỉ để báo cáo, không nuốt lỗi thật
                meta = {"__loi__": repr(exc)}
            found.append((meta_path.parent, meta))
    return found


def matches_e4(meta):
    return all(meta.get(k) == v for k, v in E4_KEYS.items())


def ckpt_for(root, fold):
    """Đường dẫn checkpoint của một fold trong `root`, hoặc None."""
    for pattern in CKPT_PATTERNS:
        hits = sorted(root.glob(pattern.format(f=fold)))
        if hits:
            return hits[0]
    return None


def find_ckpt_root(candidates, folds):
    """Thư mục nào chứa checkpoint của ĐỦ các fold cần chạy."""
    for cand in candidates:
        if not cand.exists():
            continue
        for root in [cand] + [d for d in cand.iterdir() if d.is_dir()]:
            if all(ckpt_for(root, f) is not None for f in folds):
                return root
    return None


caches = scan_caches(INPUT_ROOT)
print(f"Tìm thấy {len(caches)} cache dưới {INPUT_ROOT}:")
for path, meta in caches:
    mark = "✓ E4" if matches_e4(meta) else "  --"
    print(
        f"  {mark}  {path}\n"
        f"        crop={meta.get('crop_mode')} size={meta.get('target_size')} "
        f"align={meta.get('align_phases')}"
    )

e4 = [p for p, m in caches if matches_e4(m)]
assert e4, (
    "KHÔNG có cache nào khớp E4 trong các dataset đang mount.\n"
    f"Cần: {E4_KEYS}\n"
    "Cache đang mount là cấu hình khác (xem bảng trên). Hai đường đi:\n"
    "  1. mount đúng dataset cache E4, hoặc\n"
    "  2. build lại (~26 phút): python -m src.preprocess.build_cache "
    "--config configs/preprocess_e4.yaml"
)
CACHE_DIR = e4[0]

CKPT_ROOT = find_ckpt_root(CKPT_CANDIDATES, FOLDS)
assert CKPT_ROOT is not None, (
    f"không thấy đủ checkpoint cho fold {FOLDS} trong {CKPT_CANDIDATES}.\n"
    f"Đã thử các mẫu tên: {CKPT_PATTERNS}"
)
os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)
print("\ncache:      ", CACHE_DIR)
print("checkpoint: ", CKPT_ROOT)
for f in FOLDS:
    hit = ckpt_for(CKPT_ROOT, f)
    print(f"  fold {f}: {hit.relative_to(CKPT_ROOT)}  ({hit.stat().st_size / 2**20:.1f} MB)")

# 5 file cùng kiến trúc nên cùng kích thước — kích thước KHÔNG chứng minh chúng khác
# nhau. Băm để chắc không phải một file bị chép 5 lần với 5 cái tên.
import hashlib

digests = {}
for f in FOLDS:
    h = hashlib.sha256(ckpt_for(CKPT_ROOT, f).read_bytes()).hexdigest()[:16]
    digests[f] = h
    print(f"  fold {f} sha256: {h}")
assert len(set(digests.values())) == len(FOLDS), f"có checkpoint trùng nhau: {digests}"

## Cổng A ⚠️ — cache có đúng là E4 không

Chạy MC-dropout trên cache của E1 hay E3 sẽ **không báo lỗi gì cả**, chỉ lặng lẽ cho ra
số sai. Giống cổng ở notebook 07.

In [ ]:
import json

meta = json.loads((Path(os.environ["LLDMMRI_CACHE_DIR"]) / "cache_meta.json").read_text("utf-8"))

# Dùng lại E4_KEYS của cell trên, không chép ra bản thứ hai — hai bản sẽ trôi khỏi nhau.
for key, want in E4_KEYS.items():
    got = meta.get(key)
    assert got == want, f"cache SAI: {key} = {got!r}, cần {want!r}. Đây không phải cache E4."
assert meta["lesion_tight"]["source"] == "mask", "phải cắt theo mask, không phải bbox"

n_npz = len(list(Path(os.environ["LLDMMRI_CACHE_DIR"]).glob("*.npz")))
assert n_npz >= 498, f"chỉ có {n_npz} ca, cần 498 — cache chưa build xong"
print(f"cache_meta khớp E4 ✓ · {n_npz} ca · commit {meta.get('git_commit')}")

## Cổng B ⚠️⚠️ — model có dropout thật không

Đây là cổng quan trọng nhất của notebook. Nếu model không có lớp Dropout nào thì `K`
lượt forward cho ra `K` kết quả **giống hệt nhau**, epistemic bằng 0 khắp nơi, và bảng
kết quả vẫn in ra bình thường — một chế độ hỏng hoàn toàn im lặng.

In [ ]:
import torch

from src.eval.mc_dropout import count_dropout_modules, enable_dropout
from src.models import build_model

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

probe = build_model(CFG["model"])
n_drop = count_dropout_modules(probe)
print(f"số lớp Dropout trong model: {n_drop}")
assert n_drop > 0, (
    "model KHÔNG có lớp Dropout nào — MC-dropout sẽ không làm gì cả. "
    "Kiểm model.dropout_prob trong config."
)

# Và kiểm BatchNorm vẫn ở eval sau khi bật dropout (bẫy chính, xem docstring module).
enable_dropout(probe)
bn = [m for m in probe.modules() if isinstance(m, torch.nn.modules.batchnorm._BatchNorm)]
assert bn and not any(m.training for m in bn), "BatchNorm phải ở eval sau enable_dropout"
print(f"{len(bn)} lớp BatchNorm, tất cả ở eval ✓")
del probe

## 2. Chạy MC-dropout từng fold

`build_loaders` dựng val loader với `shuffle=False`, đúng thứ cần: các pass phải xếp ca
cùng thứ tự. `mc_dropout_predict` tự kiểm điều đó và nổ nếu lệch.

In [ ]:
import time

import numpy as np

from src.eval.mc_dropout import mc_dropout_predict, save_member_probs
from src.eval.selective import uncertainty_decomposition
from src.eval.metrics import macro_f1
from src.train.run import build_loaders

OUT_ROOT = Path(os.environ["LLDMMRI_OUTPUT_DIR"])

# Đối chiếu epoch trong checkpoint với metrics đã biết, để chắc không nạp nhầm fold.
KNOWN_EPOCH = {1: 231, 2: 297, 3: 104, 4: 135, 5: 144}

for fold in FOLDS:
    t0 = time.time()
    _, val_loader, _ = build_loaders(CFG, fold)
    model = build_model(CFG["model"]).to(DEVICE)

    state = torch.load(ckpt_for(CKPT_ROOT, fold), map_location=DEVICE)
    model.load_state_dict(state["model"])
    epoch = state.get("epoch")
    print(f"\nfold {fold}: nạp checkpoint epoch {epoch} · {len(val_loader.dataset)} ca")
    if fold in KNOWN_EPOCH and epoch != KNOWN_EPOCH[fold]:
        print(
            f"  ⚠ epoch {epoch} khác {KNOWN_EPOCH[fold]} đã ghi ở WORKLOG S-078 — "
            f"có thể đang nạp checkpoint của fold khác. Kiểm tên file."
        )
    if state.get("fold") not in (None, fold):
        raise RuntimeError(f"checkpoint ghi fold={state['fold']} nhưng đang chạy fold {fold}")

    result = mc_dropout_predict(
        model, val_loader, DEVICE, n_passes=N_PASSES,
        amp=bool(CFG["train"].get("amp", True)), seed=CFG.get("seed", 1337),
    )
    out = save_member_probs(OUT_ROOT / f"fold_{fold}" / "mc_dropout.npz", result)

    members = result["member_probs"]
    mean = members.mean(axis=0)
    unc = uncertainty_decomposition(members)
    print(
        f"  macro-F1 (trung bình {N_PASSES} lượt): {macro_f1(result['labels'], mean.argmax(1)):.4f}"
        f" · epistemic TB {unc['epistemic'].mean():.4f}"
        f" · {time.time() - t0:.0f}s -> {out.name}"
    )
    assert unc["epistemic"].max() > 1e-9, (
        f"fold {fold}: epistemic = 0 khắp nơi — dropout không thực sự chạy"
    )
    del model
    torch.cuda.empty_cache()

## 3. Gói mang về

Chỉ `.npz`, rất nhẹ. Ở máy local:

```
runs/E4_cv_results/fold_N/mc_dropout.npz
python -m src.eval.trust --run-dir runs/E4_cv_results --members
```

In [ ]:
import shutil

PACK = Path("/kaggle/working/mc_dropout_results")
shutil.rmtree(PACK, ignore_errors=True)
PACK.mkdir(parents=True)

for d in sorted(OUT_ROOT.glob("fold*")):
    src = d / "mc_dropout.npz"
    if src.exists():
        (PACK / d.name).mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, PACK / d.name / src.name)

total = sum(f.stat().st_size for f in PACK.rglob("*") if f.is_file())
print(f"đã gói {PACK}: {total / 2**20:.2f} MiB")
for f in sorted(PACK.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(PACK)}  {f.stat().st_size / 2**10:.0f} KiB")

print("""
⚠ TẢI VỀ: giải nén CHỈ MỘT LỚP. File .npz bản thân là zip; trình giải nén bung đệ quy
  sẽ biến nó thành thư mục và `src.eval.trust` sẽ không thấy (đã dính hai lần, S-078).
""")